# 03 — Classification Baselines (Phase 3)
## College Complaint Intelligence System

> **Status:** Phase 3 | Completed  
> **Experiments:**  
> - **E1:** TF-IDF (word unigrams + bigrams) + Logistic Regression (balanced)  
> - **E2:** TF-IDF + Linear SVM (Platt Calibrated via CalibratedClassifierCV)  
> **Evaluation Split:** `data/processed/val.csv` (183 examples)  
> **Note:** Held-out test split is quarantined and NOT loaded here.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.classification.train import run_classification_experiment, load_pipeline
from src.classification.predict import ComplaintClassifier
from src.data.load_data import load_config

cfg = load_config("config.yaml")
print("Project categories:", list(cfg["categories"].keys()))

### 1. Run Experiment E1: TF-IDF + Logistic Regression

In [ ]:
res_e1 = run_classification_experiment(experiment_id="E1", model_type="logreg")
m1 = res_e1["metrics"]
print(f"E1 Accuracy   : {m1['accuracy']:.4f}")
print(f"E1 Macro-F1   : {m1['macro_f1']:.4f}")
print(f"E1 Weighted-F1: {m1['weighted_f1']:.4f}")
print(f"E1 Train Time : {res_e1['training_time_s']:.3f} s")
print(f"E1 Infer Time : {res_e1['inference_ms_per_example']:.3f} ms / ex")
print("\nClassification Report:\n", m1["classification_report"])

### 2. Run Experiment E2: TF-IDF + Linear SVM (Calibrated)

In [ ]:
res_e2 = run_classification_experiment(experiment_id="E2", model_type="svm")
m2 = res_e2["metrics"]
print(f"E2 Accuracy   : {m2['accuracy']:.4f}")
print(f"E2 Macro-F1   : {m2['macro_f1']:.4f}")
print(f"E2 Weighted-F1: {m2['weighted_f1']:.4f}")
print(f"E2 Train Time : {res_e2['training_time_s']:.3f} s")
print(f"E2 Infer Time : {res_e2['inference_ms_per_example']:.3f} ms / ex")
print("\nClassification Report:\n", m2["classification_report"])

### 3. Error Analysis & Uncertainty Routing

In [ ]:
# Load errors logged during validation
errors_e1 = pd.read_csv("reports/errors_e1_logreg.csv")
print(f"Total E1 errors: {len(errors_e1)}")
errors_e1

### 4. Interactive Inference Demonstration

In [ ]:
pipe, meta = load_pipeline("models/e2_svm.joblib")
clf = ComplaintClassifier(pipe, label_names=list(cfg["categories"].keys()), confidence_threshold=0.60)

test_samples = [
    "No water in hostel block B washrooms since morning",
    "Electric spark near switchboard in classroom 204",
    "Mess dinner had stale food and sour dal",
    "General question about college annual cultural fest timing",
]
for s in test_samples:
    print(clf.predict_one(s))